In [3]:
from pathlib import Path
from bs4 import BeautifulSoup
import re
import pandas as pd

In [4]:
# Adjust the filename to match what you saved:
raw_path = Path("../data/raw/BDHSC_2024_15255_math_advanced.html")
html = raw_path.read_text(encoding="utf-8")
print(html[:500])   # sanity check: should look like HTML

<!DOCTYPE html>
<!-- saved from url=(0074)https://www.boardofstudies.nsw.edu.au/ebos/static/BDHSC_2024_12_15255.html -->
<html xmlns="http://www.w3.org/1999/xhtml" lang="en-AU" class="js"><head><meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
	
    <meta name="viewport" content="width=device-width, initial-scale=1">
    
		    
	<meta name="DC.Identifier" scheme="URI" content="http://www.boardofstudies.nsw.edu.au">     
	<meta name="DC.Title" content="Mathematics Advanced 2 un


In [5]:
%pip install lxml

Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install lxml
soup = BeautifulSoup(html, "lxml")

Note: you may need to restart the kernel to use updated packages.


In [7]:
soup = BeautifulSoup(html, "html.parser")

In [ ]:
strongs = soup.find_all("strong")
for s in strongs:
    print(repr(s.get_text()))

'Band\xa06'
'Band\xa05'
'Band\xa04'
'Band\xa03'
'Band\xa02'
'Band\xa01'


In [9]:

rows = []
for s in strongs:
    label_text = s.get_text().replace("\xa0", " ").strip()

    if not label_text.startswith("Band"):
        continue

    cell_text = s.parent.get_text().replace("\xa0", " ")

    band = re.search(r"Band\s+(\S+)", label_text).group(1)
    pct_str = re.search(r"\(([\d.]+)%\)", cell_text).group(1)

    rows.append({
        "band" : band,
        "percentage" : pct_str
    })

print(rows)

[{'band': '6', 'percentage': '22.33'}, {'band': '5', 'percentage': '27.7'}, {'band': '4', 'percentage': '27.32'}, {'band': '3', 'percentage': '17.4'}, {'band': '2', 'percentage': '4.71'}, {'band': '1', 'percentage': '0.54'}]


In [10]:
page_text = soup.get_text().replace("\xa0", " ")
m = re.search(r"Candidature\s*-\s*([\d,]+)", page_text)
candidature = int(m.group(1).replace(",", ""))
print(candidature)   # expect 16561

16561


In [11]:
df = pd.DataFrame(rows)          # 6 rows, columns: band, percentage
df["year"] = 2024
df["course_name"] = "Mathematics Advanced"
df["course_code"] = "15255"
df["candidature"] = candidature
df

,band,percentage,year,course_name,course_code,candidature
0,6,22.33,2024,Mathematics Advanced,15255,16561
1,5,27.7,2024,Mathematics Advanced,15255,16561
2,4,27.32,2024,Mathematics Advanced,15255,16561
3,3,17.4,2024,Mathematics Advanced,15255,16561
4,2,4.71,2024,Mathematics Advanced,15255,16561
5,1,0.54,2024,Mathematics Advanced,15255,16561
